# Case Study 2 — FINAL run, fully local (generator + judge on the uni GPU)

One local open generator, one local open judge, no API keys, no cost.

- **Generator:** `microsoft/phi-4` (14B, MIT licence, not gated, Microsoft family, not a reasoning model). Served with vLLM.
- **Judge:** `Qwen/Qwen2.5-72B-Instruct-AWQ` (the locked judge config). Served with vLLM.
- Both pinned to the exact Hugging Face commit, greedy decoding. Revisions are written to `results/run_manifest.json`.
- Same 215 rows, same retrieval, same prompts. All three configs by default (generation is cheap on a local model).

**Order:** generator up → generate → generator down → checkpoint zip → judge up → score → judge down → report → final zip.

**The sandbox wipes on close.** Download `results/runs_checkpoint.zip` as soon as section 7 finishes, and `results/FINAL_RESULTS.zip` at the end.
If the session dies mid-generation, rerun from the top: generation uses `--resume` and only runs the missing rows (as long as `results/runs/` still exists).

Run from the repo root.

In [14]:
!pkill -f vllm ; sleep 5 ; nvidia-smi   # kill any orphan vLLM server first

## 0. Environment probe  (must show an A6000 / 48 GB card)

In [15]:
import sys, os, subprocess, platform, urllib.request, json, time, glob, datetime
print("python:", sys.version.split()[0], "|", platform.platform())
print("cwd:", os.getcwd())
assert os.path.exists("src/run_generation.py"), "Run this notebook from the repo ROOT."
try:
    urllib.request.urlopen("https://pypi.org", timeout=5); HAS_INTERNET = True
except Exception as e:
    HAS_INTERNET = False; print("internet check failed:", e)
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | {p.total_memory/1e9:.0f} GB")
print(f"\nSUMMARY  internet={HAS_INTERNET}  gpu={HAS_GPU}")
assert HAS_GPU, "CPU-only node. Restart the server on the GPU profile (2x A6000)."

python: 3.11.13 | Linux-5.15.0-191-generic-x86_64-with-glibc2.35
cwd: /home/jovyan/case_study2
GPU 0: NVIDIA H200 NVL | 150 GB

SUMMARY  internet=True  gpu=True


## 1. Install dependencies (once)

In [16]:
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], check=False)
subprocess.run([sys.executable,"-m","pip","install","-q","openai","vllm","huggingface_hub"], check=False)
print("deps installed")


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


deps installed



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


## 2. Config — the only knobs

In [17]:
# ---------- GENERATOR (local) ----------
GEN_MODEL      = "microsoft/phi-4"
GEN_PORT       = 8001
GEN_MAX_TOKENS = 2048     # generous so truncation is never OUR cap's fault
GEN_MAX_LEN    = 8192     # context window given to vLLM (prompt ~2.5k tokens + output)

# ---------- JUDGE (local, locked) ----------
JUDGE_MODEL      = "Qwen/Qwen2.5-72B-Instruct-AWQ"
JUDGE_PORT       = 8000
JUDGE_MAX_TOKENS = 2048   # room for the per-statement verdict JSON
JUDGE_MAX_LEN    = 6144   # locked value (fits one A6000)

# ---------- RUN ----------
CONFIGS   = ["baseline1_plain_llm", "baseline2_standard_rag", "agent_structured"]
# short on GPU time? use ["agent_structured"] only
WORKERS   = 16            # concurrent requests; vLLM batches them
THRESHOLD = 0.5           # faithful cut for the bucket matrix (a sweep is reported too)
LIMIT     = None          # e.g. 8 for a quick smoke test, None for all 215

# ---------- pin exact revisions ----------
from huggingface_hub import model_info
GEN_REV   = model_info(GEN_MODEL).sha
JUDGE_REV = model_info(JUDGE_MODEL).sha
os.makedirs("results", exist_ok=True)
manifest = {
    "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
    "generator": {"model": GEN_MODEL, "revision": GEN_REV, "temperature": 0.0, "top_p": 1.0,
                  "max_tokens": GEN_MAX_TOKENS, "server": "vLLM", "max_model_len": GEN_MAX_LEN},
    "judge": {"model": JUDGE_MODEL, "revision": JUDGE_REV, "temperature": 0.0,
              "max_tokens": JUDGE_MAX_TOKENS, "server": "vLLM", "max_model_len": JUDGE_MAX_LEN},
    "configs": CONFIGS, "faithful_threshold": THRESHOLD, "limit": LIMIT,
}
json.dump(manifest, open("results/run_manifest.json","w"), indent=2)
print(json.dumps(manifest, indent=2))

def sh(cmd):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-6000:])
    if r.returncode != 0 and r.stderr: print("STDERR:\n", r.stderr[-4000:])
    return r.returncode

VLLM_ENV = {**os.environ, "VLLM_USE_FLASHINFER_SAMPLER": "0"}   # container can't JIT FlashInfer

def start_vllm(model, rev, port, max_len, util=0.90, log="vllm.log"):
    proc = subprocess.Popen([sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model, "--revision", rev, "--port", str(port), "--dtype", "auto",
        "--gpu-memory-utilization", str(util), "--max-model-len", str(max_len),
        "--served-model-name", model],
        env=VLLM_ENV, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    print(f"starting vLLM {model}@{rev[:10]} on :{port}  (log: {log})")
    for i in range(360):                      # up to 60 min for a first download
        if proc.poll() is not None:
            print("vLLM DIED. Last log lines:"); print(open(log).read()[-3000:]); return None
        try:
            urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=3)
            print(f"vLLM up after ~{i*10}s"); return proc
        except Exception:
            time.sleep(10)
    print("vLLM did not come up in time. Check", log); return proc

def stop_vllm(proc):
    if proc is None: return
    proc.terminate()
    try: proc.wait(timeout=60)
    except Exception: proc.kill()
    time.sleep(10)
    subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv"])
    print("vLLM stopped, GPU memory freed")

{
  "created_utc": "2026-09-24T15:09:56",
  "generator": {
    "model": "microsoft/phi-4",
    "revision": "2db69c1c3e91a05d2c64a3185acfbaf36f744e25",
    "temperature": 0.0,
    "top_p": 1.0,
    "max_tokens": 2048,
    "server": "vLLM",
    "max_model_len": 8192
  },
  "judge": {
    "model": "Qwen/Qwen2.5-72B-Instruct-AWQ",
    "revision": "698703eae6604af048a3d2f509995dc302088217",
    "temperature": 0.0,
    "max_tokens": 2048,
    "server": "vLLM",
    "max_model_len": 6144
  },
  "configs": [
    "baseline1_plain_llm",
    "baseline2_standard_rag",
    "agent_structured"
  ],
  "faithful_threshold": 0.5,
  "limit": null
}


## 3. Corpus (400 committed chunks)

In [18]:
CHUNKS = "results/corpus_chunks.jsonl"
print("corpus chunks:", sum(1 for _ in open(CHUNKS)))

corpus chunks: 400


## 4. Embed + index (bge, Chroma)

In [19]:
sh([sys.executable, "src/embed_and_index.py", "config/pipeline.yaml"])

$ /usr/bin/python3 src/embed_and_index.py config/pipeline.yaml
loaded 400 chunks
embedded -> (400, 768)
persisted 400 vectors -> results/chroma/eu_ai_act_guidelines



0

## 5. Retrieval check (expect Hit@5 ≈ 0.833)

In [20]:
sh([sys.executable, "src/retrieval_eval.py", "config/pipeline.yaml"])

$ /usr/bin/python3 src/retrieval_eval.py config/pipeline.yaml

Retrieval quality over 215 rows
  Hit@5   0.833   (PRIMARY)
  Hit@10  0.898
  Recall@5  0.246   Recall@10 0.374
  MRR     0.633
  wrote results/retrieval_eval.json and results/retrieval_eval.md



0

## 6. Generate with the local generator

In [21]:
gen_proc = start_vllm(GEN_MODEL, GEN_REV, GEN_PORT, GEN_MAX_LEN, util=0.90, log="vllm_generator.log")
assert gen_proc is not None
t0 = time.time()
gen_args = [sys.executable, "src/run_generation.py",
            "--provider", "openai_compatible", "--model", GEN_MODEL,
            "--base-url", f"http://localhost:{GEN_PORT}/v1", "--api-key-env", "NO_KEY_NEEDED",
            "--max-tokens", str(GEN_MAX_TOKENS), "--workers", str(WORKERS),
            "--resume", "--configs", *CONFIGS]
if LIMIT: gen_args += ["--limit", str(LIMIT)]
sh(gen_args)
print(f"generation took {(time.time()-t0)/60:.1f} min")
stop_vllm(gen_proc)

for f in sorted(glob.glob("results/runs/*.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    bad  = sum(1 for r in rows if not r["parse_ok"])
    err  = sum(1 for r in rows if (r.get("usage") or {}).get("error"))
    empty= sum(1 for r in rows if not (r.get("raw") or "").strip())
    print(f"{os.path.basename(f):32s} rows={len(rows)}  parse_fail={bad}  empty_raw={empty}  call_errors={err}")

starting vLLM microsoft/phi-4@2db69c1c3e on :8001  (log: vllm_generator.log)
vLLM up after ~70s
$ /usr/bin/python3 src/run_generation.py --provider openai_compatible --model microsoft/phi-4 --base-url http://localhost:8001/v1 --api-key-env NO_KEY_NEEDED --max-tokens 2048 --workers 16 --resume --configs baseline1_plain_llm baseline2_standard_rag agent_structured
k
  [114/210] anx3-120: pred=high-risk gold=high-risk ok
  [115/210] anx3-121: pred=high-risk gold=high-risk ok
  [116/210] anx3-122: pred=not-high-risk gold=not-high-risk ok
  [117/210] anx3-123: pred=not-high-risk gold=not-high-risk ok
  [118/210] anx3-124: pred=not-high-risk gold=not-high-risk ok
  [119/210] anx3-125: pred=not-high-risk gold=not-high-risk ok
  [120/210] anx3-126: pred=not-high-risk gold=not-high-risk ok
  [121/210] anx3-127: pred=not-high-risk gold=not-high-risk ok
  [122/210] anx3-128: pred=high-risk gold=high-risk ok
  [123/210] anx3-129: pred=high-risk gold=high-risk ok
  [124/210] anx3-130: pred=not-high-

## 7. CHECKPOINT — download `results/runs_checkpoint.zip` now

In [22]:
import shutil
shutil.make_archive("results/runs_checkpoint", "zip", "results", "runs")
print("wrote results/runs_checkpoint.zip  ->  right-click > Download in the file browser")

wrote results/runs_checkpoint.zip  ->  right-click > Download in the file browser


## 8. Judge + scoring (Qwen2.5-72B-AWQ, single card)

In [23]:
judge_proc = start_vllm(JUDGE_MODEL, JUDGE_REV, JUDGE_PORT, JUDGE_MAX_LEN, util=0.95, log="vllm_judge.log")
assert judge_proc is not None
t0 = time.time()
sh([sys.executable, "src/run_scoring.py", "--judge", "vllm",
    "--judge-model", JUDGE_MODEL, "--judge-base-url", f"http://localhost:{JUDGE_PORT}/v1",
    "--judge-max-tokens", str(JUDGE_MAX_TOKENS), "--workers", str(WORKERS),
    "--faithful-threshold", str(THRESHOLD), "--configs", *CONFIGS])
print(f"scoring took {(time.time()-t0)/60:.1f} min")
stop_vllm(judge_proc)

starting vLLM Qwen/Qwen2.5-72B-Instruct-AWQ@698703eae6 on :8000  (log: vllm_judge.log)
vLLM up after ~100s
$ /usr/bin/python3 src/run_scoring.py --judge vllm --judge-model Qwen/Qwen2.5-72B-Instruct-AWQ --judge-base-url http://localhost:8000/v1 --judge-max-tokens 2048 --workers 16 --faithful-threshold 0.5 --configs baseline1_plain_llm baseline2_standard_rag agent_structured
scoring runs in results/runs  ->  results/scoring
judge: vllm (Qwen/Qwen2.5-72B-Instruct-AWQ) @ http://localhost:8000/v1

[1/3] correctness
    agent_structured         acc=0.958  HR_f1=0.933  parse_fail=0
    baseline1_plain_llm      acc=0.833  HR_f1=0.780  parse_fail=0
    baseline2_standard_rag   acc=0.935  HR_f1=0.901  parse_fail=0

[2/3] faithfulness
  faithfulness: agent_structured (215 rows) judge=openai-compatible:Qwen/Qwen2.5-72B-Instruct-AWQ
    scored 25/215
    scored 50/215
    scored 75/215
    scored 100/215
    scored 125/215
    scored 150/215
    scored 175/215
    scored 200/215
  faithfulness: bas

## 9. Results

In [24]:
for f in ["results/scoring/correctness_summary.md",
          "results/scoring/faithfulness_summary.md",
          "results/scoring/buckets_summary.md"]:
    print("="*72); print(f); print("="*72)
    print(open(f).read() if os.path.exists(f) else "(not produced)"); print()

results/scoring/correctness_summary.md
# Correctness (predicted label vs frozen Commission ground truth)

Positive class = high-risk. Accuracy plus per-class precision/recall/F1 because the set is imbalanced (frozen). Parse failures counted as incorrect and also shown separately.

| Config | n | Accuracy | HR precision | HR recall | HR F1 | Macro F1 | Parse fails |
|---|---|---|---|---|---|---|---|
| agent_structured | 215 | 0.958 | 0.913 | 0.955 | 0.933 | 0.951 | 0 |
| baseline1_plain_llm | 215 | 0.833 | 0.653 | 0.970 | 0.780 | 0.823 | 0 |
| baseline2_standard_rag | 215 | 0.935 | 0.842 | 0.970 | 0.901 | 0.926 | 0 |

## agent_structured

Confusion (high-risk positive): tp=63 fp=6 fn=3 tn=143

By edge-case type: article-6-3-filter n=41 acc=0.927, none n=174 acc=0.966

By area: biometrics 0.938, critical-infrastructure 0.889, education 0.964, employment 1.000, essential-services 1.000, justice-democracy 0.938, law-enforcement 0.913, migration 0.957

## baseline1_plain_llm

Confusion (hig

## 10. Report + final zip — download `results/FINAL_RESULTS.zip`

In [25]:
sh([sys.executable, "src/make_report.py", "--generator", f"{GEN_MODEL} (local, vLLM)", "--max-cases", "10"])
for log in ["vllm_generator.log", "vllm_judge.log"]:
    if os.path.exists(log): shutil.copy(log, "results/")
import zipfile
with zipfile.ZipFile("results/FINAL_RESULTS.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["results/runs/*.jsonl", "results/scoring/**/*", "results/report.html",
                "results/run_manifest.json", "results/retrieval_eval.*", "results/vllm_*.log"]:
        for p in glob.glob(pat, recursive=True):
            if os.path.isfile(p): z.write(p)
print("wrote results/FINAL_RESULTS.zip  ->  download it before closing the session")

$ /usr/bin/python3 src/make_report.py --generator microsoft/phi-4 (local, vLLM) --max-cases 10
wrote results/report.html  (10 divergence cases, 3 configs)

wrote results/FINAL_RESULTS.zip  ->  download it before closing the session
